# How much data do you actually need?

A low-N benchmark for antibody property prediction. This notebook reads the
artefacts written by the scripts in `scripts/` and walks through the analysis;
it does not refit anything, so it runs in seconds.

Run `make all` first if `results/` is empty.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lown.config import RESULTS, FIGURES, TAP_TARGETS
from lown.plots import aggregate, best_head_envelope, COLORS, LABELS

pd.set_option('display.width', 140)
raw = pd.read_csv(RESULTS / 'learning_curves.csv')
raw['value'] = np.where(raw.task == 'classification', raw.get('auc'), raw.get('spearman'))
print(len(raw), 'fits')
raw.head()

## 1. What is in the two datasets

SAbDab_Chen is the large-N set: 2,409 antibodies, binary developability, 20% positive.
TAP is genuinely low-N: 241 antibodies, five continuous metrics. Both are heavy plus
light chain sequence only, no structures.

In [ ]:
pd.read_csv(RESULTS / 'dataset_summary.csv')

## 2. The leakage that a random split hides

Public antibody sets are full of near-duplicates: shared germline light chains,
affinity-matured variants of one parent, the same therapeutic under two names.
A random split puts those on both sides of the partition.

In [ ]:
pd.read_csv(RESULTS / 'leakage_summary.csv')

Clustering the whole Fv at 80% identity collapses half of SAbDab into a single
cluster, because antibody frameworks are conserved by construction. That is why the
default criterion is the union of Fv identity at 90% and CDR3 identity at 80%.

In [ ]:
pd.read_csv(RESULTS / 'cluster_summary.csv')

## 3. The baselines, before any language model

If the cheap descriptors were noise there would be nothing to compare against.
They are not noise, and the coefficients say which physical chemistry they lean on.

In [ ]:
scores = pd.read_csv(RESULTS / 'baseline_scores.csv')
scores.pivot_table(index=['dataset','target'], columns=['features','split'], values='mean').round(3)

In [ ]:
coefs = pd.read_csv(RESULTS / 'baseline_coefficients.csv')
for (d, t), g in coefs.groupby(['dataset','target'], sort=False):
    print(f"{t:<15s}", ', '.join(f'{r.feature} ({r.coefficient:+.2f})' for r in g.head(5).itertuples()))

## 4. The learning curves

Mean over 20 resampled splits, shaded band is one standard deviation. The test set
is fixed per seed; training subsamples are nested, so the N=25 set is a subset of
the N=50 set and the curve is paired rather than independent at each point.

In [ ]:
agg = aggregate(raw, 'value')
env = best_head_envelope(agg)
env.head()

In [ ]:
from IPython.display import Image, display
display(Image(str(FIGURES / 'fig1_chen_learning_curves.png')))

In [ ]:
display(Image(str(FIGURES / 'fig2_tap_learning_curves.png')))

## 5. Headline numbers

Best feature set and head per target at the largest training-set size, under each split.

In [ ]:
pd.read_csv(RESULTS / 'headline.csv').round(3)

### The honesty tax

How much of the random-split score is interpolation between near-duplicates.

In [ ]:
pd.read_csv(RESULTS / 'split_gap.csv').round(3)

In [ ]:
display(Image(str(FIGURES / 'fig3_random_vs_cluster.png')))

### Where frozen ESM-2 starts to earn its keep

In [ ]:
pd.read_csv(RESULTS / 'crossover.csv')

In [ ]:
display(Image(str(FIGURES / 'fig4_esm_vs_cheap.png')))

### Is the difference real?

Every seed uses the same split and the same nested subsample for both feature sets, so
ESM-2 minus cheap is a paired difference and a Wilcoxon signed-rank test over the 20 seeds
is the right thing to run. Comparing best-of-heads envelopes instead would smuggle in a
selection effect, so the test is run within each head.

In [ ]:
paired = pd.read_csv(RESULTS / 'paired_esm_vs_cheap.csv')
chen = paired[paired.target == 'developability']
chen.pivot_table(index=['n_request'], columns=['split','head'], values='mean_delta').round(3)

In [ ]:
# same cells, but the p-values
chen.pivot_table(index=['n_request'], columns=['split','head'], values='p_wilcoxon').round(4)

## 6. Sample efficiency

The question a wet-lab planner actually asks: how many measurements before this
is worth acting on? `NaN` means the feature set never reached the threshold at any
N available here.

In [ ]:
eff = pd.read_csv(RESULTS / 'sample_efficiency.csv')
eff[eff.split == 'cluster'].pivot(index='target', columns='features', values='n_required')

In [ ]:
display(Image(str(FIGURES / 'fig5_sample_efficiency.png')))

## 7. Cut it yourself

Everything above is a view on one tidy frame. Slice it however you like.

In [ ]:
# example: the low-N regime only, cluster-held-out, per head
low = raw[(raw.split == 'cluster') & (raw.n_train <= 100)]
low.pivot_table(index=['target','features'], columns=['head'], values='value').round(3)